# 02 — Customer Consumption · Long Term

Materialises the bronze → silver → gold tables in
[`../specifications/02-customer-consumption-long-term.md`](../specifications/02-customer-consumption-long-term.md).

**Depends on 04** — the per-zone baseline level and the intraday base/peak split are derived from the
historical `volume_forecast_silver_meter_profile`, so the long-term anchor stays consistent with the
curated short-term volumes.

**Capability tables**
- `volume_forecast_bronze_macro_drivers` (GDP / EV / heat-pump / efficiency assumptions per scenario)
- `volume_forecast_silver_consumption_lt` (monthly probabilistic structural demand)
- `volume_forecast_gold_consumption_lt_shape` (hedge-ready seasonal load shape, Cal+1..Cal+3)

**Scenario note:** three scenarios (`BASE`, `HIGH_ELECTRIFICATION`, `LOW_GROWTH`) are issued under a single
forecast `vintage_id` so the long-term curve desk can compare electrification paths and layer hedges by
P50 / P75 / P90.

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


random.seed(2)

TODAY = dt.date.today()
VINTAGE_ID = f"VF-LT-{TODAY.isoformat()}"
AS_OF_TS = dt.datetime.combine(TODAY, dt.time(8, 0))
FORECAST_YEARS = [TODAY.year + 1, TODAY.year + 2, TODAY.year + 3]
ZONES = ["DE", "NL", "FR", "BE", "AT"]

# The per-zone baseline (ZONE_BASE_MW) and intraday base/peak split (ZONE_BASE_RATIO /
# ZONE_PEAK_RATIO) are derived from 04's historical meter profile in the next cell — so the
# long-term anchor is consistent with the curated short-term volumes by construction.

# Scenario assumptions: (annual demand growth, EV penetration step, heat-pump step, efficiency drag).
SCENARIOS = {
    "BASE":                {"gdp": 1.6, "ev": 8.0,  "hp": 6.0,  "eff": 1.0},
    "HIGH_ELECTRIFICATION":{"gdp": 2.2, "ev": 16.0, "hp": 12.0, "eff": 1.2},
    "LOW_GROWTH":          {"gdp": 0.6, "ev": 4.0,  "hp": 3.0,  "eff": 0.8},
}


def season_of_month(m: int) -> str:
    if m in (12, 1, 2):
        return "WINTER"
    if m in (3, 4, 5):
        return "SPRING"
    if m in (6, 7, 8):
        return "SUMMER"
    return "AUTUMN"


def month_factor(m: int) -> float:
    # Winter-peaking demand shape (heating + lighting).
    return 1.0 + 0.18 * math.cos((m - 1) / 12.0 * 2 * math.pi)


# Historical curated profile from 04 is the baseline anchor (used in the next cell).
assert spark.catalog.tableExists(fq("volume_forecast_silver_meter_profile")), \
    "Run 04_smart_metering first — volume_forecast_silver_meter_profile is missing."

In [ ]:
# ---- Baseline from 04: per-zone average demand + intraday base/peak split ----
# Zonal demand per interval = sum of segment net-load; average by time-of-day across the
# history gives each zone's daily-average level and its trough/peak relative to that level.
_prof = spark.table(fq("volume_forecast_silver_meter_profile"))
_zint = _prof.groupBy("delivery_date", "zone_code", "interval_start").agg(F.sum("net_load_mw").alias("zload"))
_shape = _zint.groupBy("zone_code", F.date_format("interval_start", "HH:mm").alias("tod")).agg(F.avg("zload").alias("m"))
_zstats = _shape.groupBy("zone_code").agg(
    F.avg("m").alias("base_mw"), F.min("m").alias("trough_mw"), F.max("m").alias("peak_mw")).collect()
ZONE_BASE_MW = {r["zone_code"]: float(r["base_mw"]) for r in _zstats}
ZONE_BASE_RATIO = {r["zone_code"]: float(r["trough_mw"]) / float(r["base_mw"]) for r in _zstats}
ZONE_PEAK_RATIO = {r["zone_code"]: float(r["peak_mw"]) / float(r["base_mw"]) for r in _zstats}
print("Derived baseline MW:", {z: round(v, 1) for z, v in ZONE_BASE_MW.items()})
print("Base/peak ratios   :", {z: (round(ZONE_BASE_RATIO[z], 2), round(ZONE_PEAK_RATIO[z], 2)) for z in ZONE_BASE_MW})

# ---- Bronze: structural drivers per scenario / year / zone ----
macro_rows = []
for yi, year in enumerate(FORECAST_YEARS, start=1):
    for z in ZONES:
        for scn, a in SCENARIOS.items():
            macro_rows.append(Row(
                ingestion_ts=AS_OF_TS, vintage_id=VINTAGE_ID, forecast_year=year, zone_code=z, scenario=scn,
                gdp_growth_pct=round(a["gdp"] + random.gauss(0, 0.1), 2),
                ev_penetration_pct=round(a["ev"] * yi + random.gauss(0, 0.5), 2),
                heatpump_penetration_pct=round(a["hp"] * yi + random.gauss(0, 0.5), 2),
                efficiency_trend_pct=round(a["eff"], 2),
            ))
spark.createDataFrame(macro_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_macro_drivers"))

# ---- Silver: monthly probabilistic structural demand ----
lt_rows = []
for yi, year in enumerate(FORECAST_YEARS, start=1):
    for z in ZONES:
        base = ZONE_BASE_MW[z]
        for scn, a in SCENARIOS.items():
            demand_growth = (a["gdp"] - a["eff"]) / 100.0
            elec_uplift_pct = (a["ev"] + a["hp"]) / 100.0 * 0.25 * yi  # electrification adds load
            for month in range(1, 13):
                mf = month_factor(month)
                growth = (1 + demand_growth) ** yi
                p50 = base * mf * growth * (1 + elec_uplift_pct)
                uplift_mw = base * mf * growth * elec_uplift_pct
                sigma = p50 * (0.05 + 0.02 * yi)  # uncertainty grows with horizon
                lt_rows.append(Row(
                    vintage_id=VINTAGE_ID, scenario=scn, zone_code=z,
                    forecast_year=year, forecast_month=month, season=season_of_month(month),
                    p50_mw=round(p50, 2),
                    p75_mw=round(p50 + 0.6745 * sigma, 2),
                    p90_mw=round(p50 + 1.2816 * sigma, 2),
                    electrification_uplift_mw=round(uplift_mw, 2),
                ))
spark.createDataFrame(lt_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_consumption_lt"))

print("macro_drivers:", spark.table(fq("volume_forecast_bronze_macro_drivers")).count())
print("consumption_lt:", spark.table(fq("volume_forecast_silver_consumption_lt")).count())

In [ ]:
# ---- Gold: hedge-ready seasonal load shape ----
lt = spark.table(fq("volume_forecast_silver_consumption_lt"))

# Per-zone base/peak ratios derived from 04's intraday shape (computed in the previous cell).
base_ratio_map = F.create_map(*[x for z, r in ZONE_BASE_RATIO.items() for x in (F.lit(z), F.lit(r))])
peak_ratio_map = F.create_map(*[x for z, r in ZONE_PEAK_RATIO.items() for x in (F.lit(z), F.lit(r))])

shape = (lt.groupBy("vintage_id", "scenario", "zone_code", "forecast_year", "season").agg(
        F.round(F.avg("p50_mw"), 2).alias("p50_mw"),
        F.round(F.avg("p75_mw"), 2).alias("p75_mw"),
        F.round(F.avg("p90_mw"), 2).alias("p90_mw"),
    )
    .withColumn("baseload_mw", F.round(F.col("p50_mw") * base_ratio_map[F.col("zone_code")], 2))
    .withColumn("peakload_mw", F.round(F.col("p50_mw") * peak_ratio_map[F.col("zone_code")], 2))
    .withColumn("as_of_ts", F.lit(AS_OF_TS).cast("timestamp"))
    .select("vintage_id", "scenario", "zone_code", "forecast_year", "season",
            "baseload_mw", "peakload_mw", "p50_mw", "p75_mw", "p90_mw", "as_of_ts"))

shape.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_consumption_lt_shape"))
display(spark.table(fq("volume_forecast_gold_consumption_lt_shape"))
        .filter(F.col("scenario") == "HIGH_ELECTRIFICATION")
        .orderBy("zone_code", "forecast_year", "season"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_bronze_macro_drivers",
    "volume_forecast_silver_consumption_lt",
    "volume_forecast_gold_consumption_lt_shape",
]:
    print(f"  {t:46s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_02_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 02.")